# Packages


In [ ]:
'''if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")
BiocManager::install("GEOquery")'''

In [ ]:
library(Seurat)
library(GEOquery)

# Data download and mapping.csv

In [ ]:
#efetch the GEO files 
filePaths <- getGEOSuppFiles("GSE300475")

In [ ]:

gse_meta <- getGEO("GSE300475", GSEMatrix = TRUE)
pheno <- pData(gse_meta[[1]]) #phenotype/sample metadata


### mapping.csv

In [ ]:

mapping_table <- pheno[, c("geo_accession", "title", "source_name_ch1")]
print(mapping_table)

write.csv(mapping_table, file.path(data_dir, "mapping_table.csv"), row.names = FALSE)
#Used to mapping GSM to sample names in the metadata file. The mapping table is saved as a CSV file for future reference.

# Checking one sample

## Loading

In [ ]:
sample_dir <- file.path(raw_dir, "gex", "scRNAseq PT1")

raw_data <- Read10X(data.dir = sample_dir)

str(raw_data)

## Create Seurat obj and assay #

In [ ]:
# create a Seurat object from the raw data as "container"
seurat_obj <- CreateSeuratObject(
  counts   = raw_data$`Gene Expression`,
  project  = file.path(raw_dir, "gex", "scRNAseq PT1"),
  min.cells = 3,     # Genes expressed in fewer than 3 cells are filtered out
  min.features = 200 # prior to filtering, each cell must have at least 200 genes detected
)

# Add a new assay for the antibody capture data (HTO) to the Seurat object,
# ensuring that only cells present in both datasets are retained
common_cells <- intersect(colnames(seurat_obj), colnames(raw_data$`Antibody Capture`))
seurat_obj <- subset(seurat_obj, cells = common_cells)

#Hashtag Oligonucleotide
seurat_obj[["HTO"]] <- CreateAssayObject(
  counts = raw_data$`Antibody Capture`[, common_cells]
)

In [ ]:
Seurat Object
│
├── RNA assay
│
└── HTO assay

## Normalize with CLR (Centered Log-Ratio)

In [ ]:
seurat_obj <- NormalizeData(
  seurat_obj,
  assay = "HTO",
  normalization.method = "CLR" 
  #normalization method for HTO data, CLR is commonly used
)

## HTODemux


In [ ]:
 seurat_obj <- HTODemux(
  seurat_obj,
  assay = "HTO",
  positive.quantile = 0.99  
   #threshold for positive HTO signal; 
   #can be adjusted based on data characteristics
)


In [ ]:

# check the distribution of HTO classifications
table(seurat_obj$HTO_classification.global)

In [ ]:
table(seurat_obj$hash.ID)

## Visualization

In [ ]:
# Ridge plot for visualizing the distribution of HTO signals across cells
Idents(seurat_obj) <- "HTO_maxID"
RidgePlot(seurat_obj, assay = "HTO", features = rownames(seurat_obj[["HTO"]]), ncol = 3)


In [ ]:

# heatmap for visualizing the HTO signal intensities across cells
HTOHeatmap(seurat_obj, assay = "HTO")

In [ ]:
table(seurat_obj$HTO_classification.global)
table(seurat_obj$hash.ID)

In [ ]:
# Visualize the distribution of RNA counts and features across different HTO classifications
VlnPlot(seurat_obj, features = c("nCount_RNA", "nFeature_RNA"),
        group.by = "hash.ID", ncol = 2)

## Keep Singlet

In [ ]:
seurat_obj_singlet <- subset(seurat_obj, subset = HTO_classification.global == "Singlet")

# Set the active identity class to the hash ID for further analysis
Idents(seurat_obj_singlet) <- "hash.ID"

# Processing

## Def function

In [ ]:
process_sample <- function(sample_dir, sample_name) {
  raw_data <- Read10X(data.dir = sample_dir)

  obj <- CreateSeuratObject(
    counts = raw_data$`Gene Expression`,
    project = sample_name,
    min.cells = 3,
    min.features = 200
  )

  common_cells <- intersect(colnames(obj), colnames(raw_data$`Antibody Capture`))
  obj <- subset(obj, cells = common_cells)
  obj[["HTO"]] <- CreateAssayObject(counts = raw_data$`Antibody Capture`[, common_cells])

  obj <- NormalizeData(obj, assay = "HTO", normalization.method = "CLR")
  obj <- HTODemux(obj, assay = "HTO", positive.quantile = 0.99)

  obj$orig_sample <- sample_name   # tracking the original sample after merging
  return(obj)
}


## Gex root & dirs

In [ ]:

# double-check the root directory for the gene expression data
#gex organized by scripts/file_org.ipy
gex_root <- file.path(raw_dir, "gex")
sample_dirs <- list.dirs(gex_root, recursive = FALSE)
sample_names <- basename(sample_dirs)


## call process_sample function

In [ ]:

all_objects <- mapply(
  process_sample,
  sample_dir = sample_dirs,
  sample_name = sample_names,
  SIMPLIFY = FALSE #list of objects
)

## Check the HTO classification distribution for each sample

In [ ]:

lapply(names(all_objects), function(n) {
  cat("\n---", n, "---\n")
  print(table(all_objects[[n]]$HTO_classification.global))
})

# Merg and save RDS

In [ ]:
all_singlets <- lapply(all_objects, function(o) {
  subset(o, subset = HTO_classification.global == "Singlet")
})

merged_obj <- merge(
  x = all_singlets[[1]],
  y = all_singlets[-1],
  add.cell.ids = names(all_singlets)
)

saveRDS(merged_obj, file.path(proc_dir, "merged_singlets.rds"))